In [ ]:
!pip install -q librosa scipy pandas openpyxl gdown matplotlib
!pip install -q transformers
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install pytorch-tcn

In [ ]:
!git clone https://github.com/MarijaGijic/AI-SPEAK_catB.git

In [ ]:
%cd AI-SPEAK_catB

In [ ]:
!python download_data.py

In [ ]:
import os
result_dict = "/content/results" # folder za cuvanje najboljih modela
SUBMIT_DIR = "submission" # folder za cuvanje rezultata inference
os.makedirs(SUBMIT_DIR, exist_ok=True)
print("Folder napravljen:", os.path.abspath(SUBMIT_DIR))

## Ekstrakcija HuBert feature-a

In [ ]:
# ekstrakcija HuBert feature-a
from scripts.precompute_hubert import precompute_hubert

precompute_hubert()

## Ucitavanje fajlova i sanity check

In [ ]:
from src.utils.dataset import BlendshapeDataset, collate_fn_mfcc, collate_fn_hubert
from src.config import BLENDSHAPE_NAMES
from torch.utils.data import DataLoader

DATA_ROOT = 'data'
HUBERT_DIR = '/content/hubert_features'
ds = BlendshapeDataset(DATA_ROOT, speakers=['spk08', 'spk14'], load_synth=False, use_preprocessing=True, hubert_dir=HUBERT_DIR)


if len(ds) == 0:
    print('\n[!] 0 samples — checking structure:')
    import os
    for root, dirs, files in os.walk(DATA_ROOT):
        d = root.replace(DATA_ROOT, '').count(os.sep)
        if d > 3: continue
        print('  '*d + os.path.basename(root) + f'/  [{len(files)} files]')
else:
    s = ds[0]
    print(f"\nSample: {s['name']}")
    print(f"  audio_feats : {s['audio_feats'].shape}")
    print(f"  targets     : {s['targets'].shape}")
    print(f"  n phonemes  : {s['phoneme_ids'].unique().numel()}")
    print(f" is synth?    : {s['is_synth']}")
    print(f"  hubert      : {s['hubert_feats'].shape}")
    print(f"\nAll good! {len(ds)} samples ready.")

In [ ]:
result_dict = "/content/results"
SUBMIT_DIR = "submission"
os.makedirs(SUBMIT_DIR, exist_ok=True)
print("Folder napravljen:", os.path.abspath(SUBMIT_DIR))

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
from scripts.train import train

session_path = train(
    data_root=DATA_ROOT,
    model_type="tcn",
    audio_type="mfcc",

    # Model architecture
    d_model=256,
    n_channels=256,
    n_layers=5,
    dropout=0.1,
    n_speakers=2,
    use_phonemes=True,

    # Training
    lr=5e-4,
    batch_size=8,
    weight_decay=5e-5,
    patience=10,
    epochs=100,

    device='cuda',
    results_root=result_dict,
)


In [ ]:
import os, torch

print("Saved sessions:")
if os.path.exists(result_dict):
    for session in sorted(os.listdir(result_dict)):
        session_path = os.path.join(result_dict, session)
        best_model   = os.path.join(session_path, "checkpoints", "best_model.pt")
        mark = "✓" if os.path.isfile(best_model) else "✗"
        print(f"  [{mark}] {session}")
        if os.path.isfile(best_model):
            ckpt = torch.load(best_model, map_location="cpu", weights_only=False)
            print(f"        best val_loss = {ckpt.get('val_loss', float('nan')):.4f}  "
                  f"epoch = {ckpt.get('epoch', '?')}")
else:
    print(f"Results root not found: {result_dict}")